In [1]:
import arcpy
import sys
import os


arcpy.env.overwriteOutput = True

def getGDB(input_gdb):
    gdb_list= []
    arcpy.env.workspace= input_gdb
    fc_list=  arcpy.ListFeatureClasses()
    rast_list= arcpy.ListRasters()
    if fc_list:
        for fc in fc_list: gdb_list.append(fc)
    if rast_list:
        for r in rast_list: gdb_list.append(r)
    return gdb_list

def checkGDB(output_folder, output_gdb):
    if not output_gdb.endswith('.gdb'):      
        output_gdb= output_gdb+'.gdb'
        if output_folder:
            output_gdb= output_folder+'\\'+output_gdb
            return output_gdb
        else: return output_gdb
    else: 
        if output_folder:
            output_gdb= output_folder+'\\'+output_gdb
            return output_gdb
        else: return output_gdb

def checkGPKG(output_folder, output_gpkg):
    if not output_gpkg.endswith('.gpkg'):      
        output_gpkg= output_gpkg+'.gpkg'
        if output_folder:
            output_gpkg= output_folder+'\\'+output_gpkg
            return output_gpkg
        else: return output_gpkg
    else: 
        if output_folder:
            output_gpkg= output_folder+'\\'+output_gpkg
            return output_gpkg
        else: return output_gpkg
    
def setName(environment, nom_gdb, output_folder='',map_name=''):
    #l'environment és l'entorn del que surten les dades originalment. pot ser o (.gdb) o (.aprx)
    #output_gdb demana el nom de la ruta de sortida de la geodatabase resultant   
    output_gdb= checkGDB(output_folder, nom_gdb)
    arcpy.CreateFileGDB_management(output_folder,nom_gdb)
    if os.path.isfile(output_gdb) == False:
        new_list= []

        if environment.endswith('.gdb'):
            arcpy.Copy_management(environment, output_gdb)
            gdb_list= getGDB(output_gdb)
            new_list= gdb_list
            
        elif environment.endswith('.aprx'):           
            aprx= arcpy.mp.ArcGISProject(environment)
            #print('Default geodatabase:',type(aprx.defaultGeodatabase))
            #arcpy.Copy_management(aprx.defaultGeodatabase, output_gdb)  
            if map_name=='': map = aprx.listMaps()[0]
            else: map = aprx.listMaps(map_name)[0]
            layers= map.listLayers()
            arcpy.env.workspace= aprx.defaultGeodatabase
            for fc in arcpy.ListFeatureClasses():
                print(fc)
                for lyrx in layers: 
                    if lyrx.isFeatureLayer or lyrx.isRasterLayer:
                        if lyrx.name==fc:
                            print(lyrx.name)
                            new_list.append(lyrx.name)
                            arcpy.conversion.FeatureClassToGeodatabase(fc,output_gdb)
                    elif lyrx.isBasemapLayer: continue

            gdb_list= getGDB(aprx.defaultGeodatabase)
            #export layer package of many layers
            #if map_name!='': arcpy.PackageLayer_management(layers, output_folder+'\\'+map_name+'.lpkx')
            #else: arcpy.PackageLayer_management(layers, output_folder+'\\'+nom_gdb+'.lpkx')
        
        arcpy.env.workspace= output_gdb
        print('Llista de la geodatabase:\n',gdb_list)
        print('Nova_Llista:\n',new_list)
        
        if True:
            
            #rename the layers in the gdb
            n=0
            for i in new_list:
                for file in gdb_list:
                    if i == file:
                        n+=1
                        if n<10:
                            print('change ->',file)
                            arcpy.Rename_management(file, '_0'+str(n)+'_'+file)
                            print('New FC:', '_0'+str(n)+'_'+i)
                        else:
                            print('change ->',file)
                            arcpy.Rename_management(file, '_'+str(n)+'_'+file)
                            print('New FC:', '_'+str(n)+'_'+i)      

            gdb_features= arcpy.ListFeatureClasses()
            gdb_rasters= arcpy.ListRasters()
            
            #export rasters into a subfolder
            raster_folder= output_folder+'\\rasters'
            if os.path.isdir(raster_folder): pass
            else: os.mkdir(raster_folder)

            for r in gdb_rasters:
                arcpy.management.CopyRaster(r,raster_folder+'\\'+r+'.tif',None,None,-9999,None,None,'32_BIT_FLOAT',None,None,'TIFF',None)
            
            output_gpkg= checkGPKG(output_folder, nom_gdb)
            if os.path.isfile(output_gpkg) == False:
                arcpy.management.CreateSQLiteDatabase(output_gpkg,'GEOPACKAGE_1.3' )
                for fc in gdb_features:
                    arcpy.management.CopyFeatures(output_gdb+'\\'+fc,output_gpkg+'\\'+fc)
                print('GPKG creat')       

    else: print('GDB ',output_gdb, ' already exists')

if __name__ == "__main__":

    env = R'C:\Users\becari.alex.marcos\Documents\ArcGIS\Projects\testing\testing.aprx'
    carpeta = R'C:\Users\becari.alex.marcos\Documents\ArcGIS\Projects\testing'
    mapa = 'Map3'
    nom_gdb= 'Calix'
    
    setName(env, nom_gdb, carpeta, mapa)

Unitats2_ExportFeatures
Plistocè_resto
Plistocè_resto
gt125mv20sh0fbp1r01_Dissolve
gt125mv20sh0fbp1r01_Dissolve_canvis
Llista de la geodatabase:
 ['Unitats2_ExportFeatures', 'Plistocè_resto', 'gt125mv20sh0fbp1r01_Dissolve', 'gt125mv20sh0fbp1r01_Dissolve_canvis']
Nova_Llista:
 ['Plistocè_resto']
change -> Plistocè_resto
New FC: _01_Plistocè_resto
GPKG creat
